In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import joblib
import os

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop ID column before processing; zero variance / non-predictive asset
df = df.drop(columns=['customerID'])

# Fill the ~11 nulls in TotalCharges with 0 (new customers with no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('Shape after cleaning:', df.shape)
print('Remaining nulls:', df.isnull().sum().sum())

Shape after cleaning: (7043, 20)
Remaining nulls: 0


In [3]:
# Feature engineering
# These derived features add signal without leaking future data

# Avg monthly spend over their tenure
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)

# Number of add-on services subscribed (proxy for engagement)
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['NumServices'] = df[service_cols].apply(
    lambda row: sum(1 for v in row if v not in ['No', 'No internet service', 'No phone service']),
    axis=1
)

# Binary: is the customer on a long-term contract?
df['LongTermContract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

print('New features: AvgMonthlySpend, NumServices, LongTermContract')
df[['AvgMonthlySpend', 'NumServices', 'LongTermContract']].describe()

New features: AvgMonthlySpend, NumServices, LongTermContract


,AvgMonthlySpend,NumServices,LongTermContract
count,7043.000000,7043.000000,7043.000000
mean,58.990789,4.146244,0.449808
std,30.579745,2.312720,0.497510
min,0.000000,1.000000,0.000000
25%,26.041493,2.000000,0.000000
50%,60.937879,4.000000,0.000000
75%,84.830742,6.000000,1.000000
max,118.969863,9.000000,1.000000


In [4]:
# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

print('Features:', X.shape)
print('Target balance:', y.value_counts().to_dict())

Features: (7043, 22)
Target balance: {0: 5174, 1: 1869}


In [5]:
# Identify column types
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric:', numeric_cols)
print('Categorical:', categorical_cols)

Numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'NumServices', 'LongTermContract']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\Neel\AppData\Local\Temp\ipykernel_6768\2149819570.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [6]:
# Build a scikit-learn ColumnTransformer
# This is the 'production-ready' way to do preprocessing — it prevents data leakage
# because the scaler is fit only on training data

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

print('Preprocessor built.')

Preprocessor built.


In [7]:
# Stratified split — preserves churn ratio in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%}')
print(f'Test churn rate:  {y_test.mean():.1%}')

Train: (5634, 22), Test: (1409, 22)
Train churn rate: 26.5%
Test churn rate:  26.5%


In [ ]:
# Save preprocessed splits so notebook 3 can pick up here
os.makedirs('../data', exist_ok=True)

# Quick sanity check on stratify balance before serialization
# print("Train target mix:\\n", y_train.value_counts(normalize=True))
# print("Test target mix:\\n", y_test.value_counts(normalize=True))

X_train.to_parquet('../data/X_train.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

# Save preprocessor for reuse in production-style pipelines
joblib.dump(preprocessor, '../data/preprocessor.pkl')
print('Saved splits and preprocessor.')

Saved splits and preprocessor.
